# AgroVision — No-OOM v4
## รวมข้อดีของ v3 (No-OOM) + v2 (CutMix, OOF, TTA) ไว้ในเวอร์ชันเดียว

### ✅ OOM Fixes ทั้งหมด:
| Fix | รายละเอียด |
|-----|------------|
| `memory_growth=True` | ป้องกัน TF จอง VRAM เต็มตั้งแต่ต้น |
| `BATCH_SIZE = 8` | ลดจาก 16 → 8 ประหยัด VRAM ~50% |
| `gc.collect()` | คืน RAM Python หลังทุก fold |
| `clear_session()` + `del` | ล้าง TF graph & weights ออกจาก VRAM |
| Pseudo-label sequential load | โหลด 1 model ทีละตัว |
| TTA ×5 | ลดจาก ×10 เพื่อประหยัด memory |

### 🚀 Features จาก v2:
- CutMix + MixUp, Heavy augmentation (random crop, hue)
- OOF (Out-of-Fold) predictions tracking
- Pseudo-labeling (high-confidence test images)
- TTA ×5, Confusion matrix, Classification report

**คาดว่าได้: 85-90%**

In [ ]:
# ══════════════════════════════════════════════════════════
# CONFIG — No-OOM v4
# ══════════════════════════════════════════════════════════
IMG_SIZE        = 320
BATCH_SIZE      = 8       # FIX: 16->8 (ป้องกัน OOM)
N_FOLDS         = 5
EPOCHS_HEAD     = 5
EPOCHS_FINE     = 35
EPOCHS_PSEUDO   = 8
LR_HEAD         = 5e-4
LR_FINE         = 2e-5    # FIX: 3e-6->2e-5 (สำคัญมาก!)
LABEL_SMOOTH    = 0.1
PATIENCE        = 10      # FIX: 7->10
PSEUDO_THRESH   = 0.90
SEED            = 42
BACKBONE        = 'EfficientNetV2M'
N_TTA           = 5       # FIX: 10->5 (ประหยัด memory)

DATA_DIR = '/kaggle/input/competitions/cassava-leaf-disease-classification'
OUT_DIR  = '/kaggle/working'
print('Config OK')


In [ ]:
import os, json, random, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf

# ✅ FIX 1: Memory growth — ป้องกัน TF จอง VRAM เต็ม
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
print(f'GPUs: {len(gpus)} found')

# Mixed Precision — ประหยัด VRAM ~50%
tf.keras.mixed_precision.set_global_policy('mixed_float16')
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f'TF {tf.__version__} | Policy: {tf.keras.mixed_precision.global_policy()}')


In [ ]:
with open(f'{DATA_DIR}/label_num_to_disease_map.json') as f:
    LABEL_MAP = {int(k): v.replace('Cassava ', '') for k, v in json.load(f).items()}
NUM_CLASSES = len(LABEL_MAP)
CLASS_NAMES = [LABEL_MAP[i] for i in range(NUM_CLASSES)]

df = pd.read_csv(f'{DATA_DIR}/train.csv')
df['filepath'] = df['image_id'].apply(lambda x: f'{DATA_DIR}/train_images/{x}')
print(f'Classes: {CLASS_NAMES}')
print(f'Total: {len(df)} images')
print(df['label'].map(LABEL_MAP).value_counts())


In [ ]:
# ── Data Pipeline ─────────────────────────────────────────────
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32)
    label = tf.one_hot(label, NUM_CLASSES)
    return img, label

def augment_heavy(img, label):
    """Heavy augmentation from v2"""
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, 0.3)
    img = tf.image.random_contrast(img, 0.7, 1.3)
    img = tf.image.random_saturation(img, 0.7, 1.3)
    img = tf.image.random_hue(img, 0.1)
    k = tf.random.uniform([], 0, 4, dtype=tf.int32)
    img = tf.image.rot90(img, k)
    # Random crop + resize (simulates zoom)
    crop_size = tf.random.uniform([], int(IMG_SIZE * 0.8), IMG_SIZE, dtype=tf.int32)
    img = tf.image.random_crop(img, [crop_size, crop_size, 3])
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.clip_by_value(img, 0, 255)
    return img, label

def mixup_batch(images, labels, alpha=0.3):
    lam = np.float32(np.random.beta(alpha, alpha))
    idx = tf.random.shuffle(tf.range(tf.shape(images)[0]))
    return (lam * images + (1 - lam) * tf.gather(images, idx),
            lam * labels + (1 - lam) * tf.gather(labels, idx))

def cutmix_batch(images, labels, alpha=1.0):
    """CutMix — memory-safe: numpy mask computed once per batch"""
    lam = np.float32(np.random.beta(alpha, alpha))
    idx = tf.random.shuffle(tf.range(tf.shape(images)[0]))
    cut = int(IMG_SIZE * np.sqrt(1 - lam))
    cx = random.randint(0, IMG_SIZE)
    cy = random.randint(0, IMG_SIZE)
    x1, x2 = max(cx - cut // 2, 0), min(cx + cut // 2, IMG_SIZE)
    y1, y2 = max(cy - cut // 2, 0), min(cy + cut // 2, IMG_SIZE)
    lam_actual = 1.0 - ((x2 - x1) * (y2 - y1) / (IMG_SIZE * IMG_SIZE))
    # ✅ FIX: numpy mask สร้างครั้งเดียว ไม่ใช้ tensor_scatter (ป้องกัน OOM)
    mask = np.ones((1, IMG_SIZE, IMG_SIZE, 1), dtype=np.float32)
    mask[:, y1:y2, x1:x2, :] = 0
    mask_t = tf.constant(mask)
    shuffled = tf.gather(images, idx)
    mixed = images * mask_t + shuffled * (1.0 - mask_t)
    mixed_labels = lam_actual * labels + (1.0 - lam_actual) * tf.gather(labels, idx)
    return mixed, mixed_labels

def build_dataset(df_sub, training=False, use_cutmix=False):
    ds = tf.data.Dataset.from_tensor_slices(
        (df_sub['filepath'].values, df_sub['label'].values.astype('int32'))
    )
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.shuffle(2000, seed=SEED)  # ✅ FIX: 3000->2000
        ds = ds.map(augment_heavy, num_parallel_calls=AUTOTUNE)
        ds = ds.batch(BATCH_SIZE)
        if use_cutmix:
            ds = ds.map(
                lambda x, y: tf.cond(
                    tf.random.uniform([]) > 0.5,
                    lambda: mixup_batch(x, y),
                    lambda: cutmix_batch(x, y)
                ), num_parallel_calls=AUTOTUNE)
        else:
            ds = ds.map(mixup_batch, num_parallel_calls=AUTOTUNE)
    else:
        ds = ds.batch(BATCH_SIZE)
    return ds.prefetch(AUTOTUNE)

print('Pipeline ready')


In [ ]:
# ── Model Builder ─────────────────────────────────────────────
def build_model(trainable_backbone=False):
    backbone = getattr(tf.keras.applications, BACKBONE)(
        include_top=False, weights='imagenet',
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    backbone.trainable = trainable_backbone
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = backbone(inputs, training=False)
    avg = tf.keras.layers.GlobalAveragePooling2D()(x)
    mx  = tf.keras.layers.GlobalMaxPooling2D()(x)
    x   = tf.keras.layers.Concatenate()([avg, mx])
    x   = tf.keras.layers.BatchNormalization()(x)
    x   = tf.keras.layers.Dropout(0.4)(x)
    x   = tf.keras.layers.Dense(512, activation='swish',
              kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x   = tf.keras.layers.Dropout(0.3)(x)
    out = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)
    return tf.keras.Model(inputs, out), backbone

def compile_model(model, lr, steps_per_epoch=None, warmup_epochs=2):
    if steps_per_epoch:
        total  = EPOCHS_FINE * steps_per_epoch
        warmup = warmup_epochs * steps_per_epoch
        lr_sched = tf.keras.optimizers.schedules.CosineDecay(
            initial_learning_rate=lr,
            decay_steps=max(total - warmup, 1),
            alpha=1e-7,
        )
    else:
        lr_sched = lr
    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr_sched, clipnorm=1.0),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH),
        metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=2, name='top2')]
    )

print('Model builder ready')


In [ ]:
# ══════════════════════════════════════════════════════════
# 5-FOLD CROSS VALIDATION — No-OOM v4
# ══════════════════════════════════════════════════════════
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_val_accs = []
oof_preds = np.zeros((len(df), NUM_CLASSES))  # OOF predictions

for fold, (train_idx, val_idx) in enumerate(skf.split(df, df['label'])):
    print(f'\n' + '='*60)
    print(f'  FOLD {fold+1}/{N_FOLDS}')
    print('='*60)

    train_df = df.iloc[train_idx].reset_index(drop=True)
    val_df   = df.iloc[val_idx].reset_index(drop=True)

    cw = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=train_df['label'].values)
    class_weight = {i: float(w) for i, w in enumerate(cw)}
    steps = len(train_df) // BATCH_SIZE

    train_ds = build_dataset(train_df, training=True, use_cutmix=True)  # CutMix+MixUp
    val_ds   = build_dataset(val_df,   training=False)

    # Phase 1: Head only
    model, backbone = build_model(trainable_backbone=False)
    compile_model(model, LR_HEAD)
    model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD,
              class_weight=class_weight, verbose=1)

    # Phase 2: Fine-tune
    backbone.trainable = True
    compile_model(model, LR_FINE, steps_per_epoch=steps)

    ckpt = f'{OUT_DIR}/fold{fold+1}_best.keras'
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(ckpt, monitor='val_accuracy',
                                            save_best_only=True, verbose=1),
        tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=PATIENCE,
                                          restore_best_weights=True, verbose=1),
        tf.keras.callbacks.CSVLogger(f'{OUT_DIR}/fold{fold+1}_log.csv'),
    ]

    hist = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FINE,
                     class_weight=class_weight, callbacks=callbacks, verbose=1)

    best_acc = max(hist.history['val_accuracy'])
    fold_val_accs.append(best_acc)
    print(f'Fold {fold+1} best val_acc: {best_acc*100:.2f}%')

    # OOF predictions from best checkpoint
    best_model = tf.keras.models.load_model(ckpt, compile=False)
    oof_probs = best_model.predict(val_ds, verbose=0)
    oof_preds[val_idx] = oof_probs
    del best_model

    # ✅ FIX 2: Full memory cleanup
    del model, backbone, train_ds, val_ds
    tf.keras.backend.clear_session()
    gc.collect()

print('\n' + '='*60)
print('5-Fold CV Results:')
for i, acc in enumerate(fold_val_accs):
    print(f'  Fold {i+1}: {acc*100:.2f}%')
if fold_val_accs:
    print(f'  Mean: {np.mean(fold_val_accs)*100:.2f}% +/- {np.std(fold_val_accs)*100:.2f}%')
print('='*60)


In [ ]:
# ── OOF Ensemble Accuracy + Confusion Matrix ──────────────────
oof_pred_labels = np.argmax(oof_preds, axis=1)
true_labels     = df['label'].values
oof_acc = np.mean(oof_pred_labels == true_labels)
print(f'OOF Ensemble Accuracy: {oof_acc*100:.2f}%')
print()
print(classification_report(true_labels, oof_pred_labels, target_names=CLASS_NAMES))

cm = confusion_matrix(true_labels, oof_pred_labels)
plt.figure(figsize=(8, 6))
sns.heatmap(cm / cm.sum(axis=1, keepdims=True), annot=True, fmt='.2f',
            cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title(f'OOF Ensemble Accuracy: {oof_acc*100:.1f}%')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/confusion_matrix.png', dpi=150)
plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════
# PSEUDO-LABELING — Memory-safe (load 1 model at a time)
# ✅ FIX: ไม่โหลด 5 models พร้อมกัน
# ══════════════════════════════════════════════════════════
test_images  = sorted(glob.glob(f'{DATA_DIR}/test_images/*.jpg'))
print(f'Test images found: {len(test_images)}')
df_augmented = df.copy()

if len(test_images) > 0:
    test_df    = pd.DataFrame({'filepath': test_images, 'label': [0]*len(test_images)})
    test_preds = np.zeros((len(test_images), NUM_CLASSES))
    loaded_n   = 0

    for fold_i in range(N_FOLDS):
        ckpt = f'{OUT_DIR}/fold{fold_i+1}_best.keras'
        if not os.path.exists(ckpt):
            continue
        print(f'  Predicting fold {fold_i+1}...', end=' ')
        test_ds = build_dataset(test_df, training=False)
        m = tf.keras.models.load_model(ckpt, compile=False)
        test_preds += m.predict(test_ds, verbose=0)
        loaded_n += 1
        # ✅ FIX: clear ทันทีหลัง predict ทุกครั้ง
        del m, test_ds
        tf.keras.backend.clear_session()
        gc.collect()
        print('done')

    if loaded_n > 0:
        test_preds /= loaded_n
        max_conf    = test_preds.max(axis=1)
        pseudo_mask = max_conf >= PSEUDO_THRESH
        pseudo_df   = test_df[pseudo_mask].copy()
        pseudo_df['label'] = test_preds[pseudo_mask].argmax(axis=1).astype(int)
        n_pseudo = len(pseudo_df)
        n_real   = len(df)
        print(f'Pseudo-labels (conf>={PSEUDO_THRESH}): {n_pseudo}/{len(test_images)}')
        print(pseudo_df['label'].map(LABEL_MAP).value_counts())
        df_augmented = pd.concat([df, pseudo_df[['filepath', 'label']]], ignore_index=True)
        print(f'Augmented dataset: {len(df_augmented)} total ({n_real} real + {n_pseudo} pseudo)')
else:
    print('No test images — skipping pseudo-labeling')


In [ ]:
# ── Final Model: retrain best fold on full data + pseudo labels ───
best_fold = int(np.argmax(fold_val_accs))
best_acc_pct = fold_val_accs[best_fold] * 100
print(f'Best fold: {best_fold+1} ({best_acc_pct:.2f}%)')
print('Retraining on full dataset + pseudo labels...')

full_train_ds = build_dataset(df_augmented, training=True, use_cutmix=True)
cw_full = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES),
                                y=df_augmented['label'].values)
cw_full = {i: float(w) for i, w in enumerate(cw_full)}

final_model = tf.keras.models.load_model(
    f'{OUT_DIR}/fold{best_fold+1}_best.keras', compile=False)
final_model.trainable = True
final_model.compile(
    optimizer=tf.keras.optimizers.Adam(LR_FINE * 0.5, clipnorm=1.0),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH),
    metrics=['accuracy']
)

final_model.fit(
    full_train_ds, epochs=EPOCHS_PSEUDO, class_weight=cw_full,
    callbacks=[
        tf.keras.callbacks.ModelCheckpoint(
            f'{OUT_DIR}/final_model.keras',
            monitor='accuracy', save_best_only=True, verbose=1),
        tf.keras.callbacks.CSVLogger(f'{OUT_DIR}/final_log.csv'),
    ]
)
del full_train_ds
gc.collect()
print('Final model done!')


In [ ]:
# ══════════════════════════════════════════════════════════
# TTA x N_TTA — Memory-safe (running average per batch)
# ✅ FIX: ไม่ accumulate predictions ทั้ง dataset ใน RAM
# ══════════════════════════════════════════════════════════
_, val_idx_best = list(skf.split(df, df['label']))[best_fold]
val_df_best = df.iloc[val_idx_best].reset_index(drop=True)
val_ds_tta  = build_dataset(val_df_best, training=False)

final_model = tf.keras.models.load_model(
    f'{OUT_DIR}/final_model.keras', compile=False)

tta_preds_accum = np.zeros((len(val_df_best), NUM_CLASSES), dtype=np.float32)

for tta_i in range(N_TTA):
    print(f'TTA pass {tta_i+1}/{N_TTA}...', end=' ')
    batch_preds = []
    for imgs, _ in val_ds_tta:
        # ✅ FIX: augment + predict ทีละ batch (ไม่เก็บ imgs ทั้งหมดใน RAM)
        aug = tf.image.random_flip_left_right(imgs)
        aug = tf.image.random_flip_up_down(aug)
        aug = tf.image.random_brightness(aug, 0.1)
        k   = tf.random.uniform([], 0, 4, dtype=tf.int32)
        aug = tf.image.rot90(aug, k)
        preds = final_model(aug, training=False).numpy()
        batch_preds.append(preds)
    tta_preds_accum += np.concatenate(batch_preds) / N_TTA
    del batch_preds
    gc.collect()
    print('done')

tta_labels   = val_df_best['label'].values
tta_acc      = np.mean(np.argmax(tta_preds_accum, axis=1) == tta_labels)
no_tta_preds = final_model.predict(val_ds_tta, verbose=0)
no_tta_acc   = np.mean(np.argmax(no_tta_preds, axis=1) == tta_labels)

print(f'Val Accuracy (no TTA):   {no_tta_acc*100:.2f}%')
print(f'Val Accuracy (TTA x{N_TTA}):  {tta_acc*100:.2f}%')
improvement = (tta_acc - no_tta_acc) * 100
print(f'TTA improvement: +{improvement:.2f}%')


In [ ]:
# ── Export ────────────────────────────────────────────────────
with open(f'{OUT_DIR}/cassava_labels.json', 'w') as f:
    json.dump({str(k): v for k, v in LABEL_MAP.items()}, f, indent=2)

# TFLite FP16
converter = tf.lite.TFLiteConverter.from_keras_model(final_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite = converter.convert()
with open(f'{OUT_DIR}/cassava_model_v4_fp16.tflite', 'wb') as f:
    f.write(tflite)

summary = {
    'version'          : 'No-OOM v4',
    'backbone'         : BACKBONE,
    'batch_size'       : BATCH_SIZE,
    'img_size'         : IMG_SIZE,
    'n_folds'          : N_FOLDS,
    'fold_val_accs'    : [round(x*100, 2) for x in fold_val_accs],
    'mean_fold_acc'    : round(float(np.mean(fold_val_accs))*100, 2),
    'oof_ensemble_acc' : round(float(oof_acc)*100, 2),
    'final_val_no_tta' : round(float(no_tta_acc)*100, 2),
    'final_val_tta'    : round(float(tta_acc)*100, 2),
    'oom_fixes'        : [
        'memory_growth=True', 'BATCH_SIZE=8', 'gc.collect() per fold',
        'clear_session() per fold', 'pseudo_sequential_load', 'TTA_x5'
    ],
    'features'         : [
        'CutMix+MixUp', 'HeavyAugmentation', 'OOF-Tracking',
        'PseudoLabeling', 'TTA-x5', 'CosineDecay+Warmup', 'ClassWeights'
    ]
}
with open(f'{OUT_DIR}/training_summary_v4.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print('\nFILES TO DOWNLOAD:')
print('  final_model.keras')
print('  cassava_labels.json')
print('  cassava_model_v4_fp16.tflite')
print('  confusion_matrix.png')
